In [0]:
# --- 1. DEFINICIÓN DE RUTAS (Unity Catalog) ---
# Ajusta el nombre del volumen si es diferente
base_volume = "/Volumes/ecomotive/landing/ecomotive_vol"

# Rutas separadas para evitar bucles infinitos
path_inbox       = f"{base_volume}/inbox"
path_checkpoints = f"{base_volume}/sys/checkpoints/bronze_sensors"
path_schemas     = f"{base_volume}/sys/schemas/bronze_sensors"
target_table     = "ecomotive.landing.bronze_sensors"

# --- 2. LIMPIEZA TOTAL (Para empezar de cero) ---
spark.sql(f"DROP TABLE IF EXISTS {target_table}")
dbutils.fs.rm(path_inbox, True)       # Borramos datos viejos
dbutils.fs.rm(path_checkpoints, True) # Borramos memoria de Spark
dbutils.fs.rm(path_schemas, True)     # Borramos esquemas inferidos
dbutils.fs.mkdirs(path_inbox)         # Recreamos la carpeta de entrada

print(f"Entorno limpio. Rutas configuradas en: {base_volume}")

In [0]:
# Datos iniciales (3 camiones)
json_batch_01 = """
{"id_camion": 101, "velocidad": 80, "rpm": 2000, "timestamp": "2023-01-01T10:00:00"}
{"id_camion": 102, "velocidad": 75, "rpm": 1800, "timestamp": "2023-01-01T10:00:00"}
{"id_camion": 103, "velocidad": 90, "rpm": 2200, "timestamp": "2023-01-01T10:00:00"}
"""

# Escribimos el archivo en la carpeta 'inbox' que vigilará Auto Loader
dbutils.fs.put(f"{path_inbox}/sensor_data_batch_01.json", json_batch_01.strip(), overwrite=True)

print("Batch 01 generado en la Inbox.")

In [0]:
# Ver contenido del Volumen
files = dbutils.fs.ls(path_inbox)
display(files)

In [0]:
# 1. LECTURA (ReadStream)
df_stream = (spark.readStream
  .format("cloudFiles") # Autoloader
  .option("cloudFiles.format", "json") # Formato
  .option("cloudFiles.inferColumnTypes", "true")
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Evolucion del Esquema #failOnNewColumns #rescue
  .option("cloud_files.useNotifications", "false") # Detección de Cambios
  .option("cloudFiles.schemaLocation", path_schemas)         # Guardar esquema
  .load(path_inbox)
)

# 2. ESCRITURA (WriteStream)
query = (df_stream.writeStream
  .format("delta")
  .option("checkpointLocation", path_checkpoints) # Memoria del stream
  .option("mergeSchema", "true")
  .trigger(availableNow=True) # Procesa lo que hay y se apaga (Batch eficiente)
  .table(target_table)
)

query.awaitTermination()
print(f"Ingesta Inicial completada en: {target_table}")

In [0]:
%sql

SELECT * FROM ecomotive.landing.bronze_sensors 
ORDER BY timestamp, id_camion;

In [0]:
# --- BATCH 02 (11:00 AM) ---
json_batch_02 = """
{"id_camion": 101, "velocidad": 82, "rpm": 2100, "timestamp": "2023-01-01T11:00:00"}
{"id_camion": 102, "velocidad": 72, "rpm": 1700, "timestamp": "2023-01-01T11:00:00"}
"""
dbutils.fs.put(f"{path_inbox}/sensor_data_batch_02.json", json_batch_02.strip(), overwrite=True)

# --- BATCH 03 (12:00 PM) ---
json_batch_03 = """
{"id_camion": 101, "velocidad": 85, "rpm": 2150, "timestamp": "2023-01-01T12:00:00"}
{"id_camion": 103, "velocidad": 95, "rpm": 2300, "timestamp": "2023-01-01T12:00:00"}
"""
dbutils.fs.put(f"{path_inbox}/sensor_data_batch_03.json", json_batch_03.strip(), overwrite=True)

print("Batch 02 y 03 generados. Listos para ingestión.")

In [0]:
# Ver contenido del Volumen
files = dbutils.fs.ls(path_inbox)
display(files)